In [10]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sktechno710/aggregate-churn-data-made-from-collabspace-data/churn_prob_data.csv
/kaggle/input/datasets/sktechno710/aggregate-churn-data-made-from-collabspace-data/aggregate_churn.csv


### Data Script

In [20]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Assuming you have 500 unique users from Day 1
user_ids = np.arange(0, 500)

# 60% Basic ($15/mo), 30% Pro ($50/mo), 10% Enterprise ($200/mo)
tiers = np.random.choice(['Basic', 'Pro', 'Enterprise'], size=500, p=[0.6, 0.3, 0.1])
fees = np.where(tiers == 'Basic', 15, np.where(tiers == 'Pro', 50, 200))

# Gross margin mapping (Enterprise has higher support costs, thus slightly lower margin percentage)
margins = np.where(tiers == 'Basic', 0.90, np.where(tiers == 'Pro', 0.85, 0.75))

df = pd.DataFrame({
    'user_id': user_ids,
    'subscription_tier': tiers,
    'monthly_fee': fees,
    'gross_margin_pct': margins
})

print("Financial data generated successfully: user_subscriptions.csv")

Financial data generated successfully: user_subscriptions.csv


In [21]:
df

,user_id,subscription_tier,monthly_fee,gross_margin_pct
0,0,Basic,15,0.90
1,1,Enterprise,200,0.75
2,2,Pro,50,0.85
3,3,Basic,15,0.90
4,4,Basic,15,0.90
...,...,...,...,...
495,495,Basic,15,0.90
496,496,Basic,15,0.90
497,497,Basic,15,0.90
498,498,Enterprise,200,0.75


### Financial Data Integration

In [23]:
churn_prob_data = pd.read_csv('/kaggle/input/datasets/sktechno710/aggregate-churn-data-made-from-collabspace-data/churn_prob_data.csv')


0        0
1        1
2        2
3        3
4        4
      ... 
495    495
496    496
497    497
498    498
499    499
Name: Unnamed: 0, Length: 500, dtype: int64

In [33]:
merged_df = pd.merge(df, churn_prob_data, left_index=True, right_index=True)
merged_df

,user_id,subscription_tier,monthly_fee,gross_margin_pct,Unnamed: 0,churn
0,0,Basic,15,0.90,0,0.81
1,1,Enterprise,200,0.75,1,0.00
2,2,Pro,50,0.85,2,0.99
3,3,Basic,15,0.90,3,0.08
4,4,Basic,15,0.90,4,1.00
...,...,...,...,...,...,...
495,495,Basic,15,0.90,495,0.00
496,496,Basic,15,0.90,496,0.00
497,497,Basic,15,0.90,497,0.74
498,498,Enterprise,200,0.75,498,0.07


In [36]:
merged_df = merged_df[['user_id','subscription_tier','monthly_fee','gross_margin_pct','churn']]
merged_df.reset_index(drop=True, inplace=True)
merged_df

,user_id,subscription_tier,monthly_fee,gross_margin_pct,churn
0,0,Basic,15,0.90,0.81
1,1,Enterprise,200,0.75,0.00
2,2,Pro,50,0.85,0.99
3,3,Basic,15,0.90,0.08
4,4,Basic,15,0.90,1.00
...,...,...,...,...,...
495,495,Basic,15,0.90,0.00
496,496,Basic,15,0.90,0.00
497,497,Basic,15,0.90,0.74
498,498,Enterprise,200,0.75,0.07


In [63]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   user_id            500 non-null    int64  
 1   subscription_tier  500 non-null    object 
 2   monthly_fee        500 non-null    int64  
 3   gross_margin_pct   500 non-null    float64
 4   churn              500 non-null    float64
 5   projected_cltv     500 non-null    float64
dtypes: float64(3), int64(2), object(1)
memory usage: 23.6+ KB


### Indivdual CLTV Calculation

In [40]:
def projected_cltv(row):
    return row['monthly_fee']/max(0.1,row['churn'])
merged_df['projected_cltv'] = merged_df.apply(projected_cltv,axis=1)
merged_df

,user_id,subscription_tier,monthly_fee,gross_margin_pct,churn,projected_cltv
0,0,Basic,15,0.90,0.81,18.518519
1,1,Enterprise,200,0.75,0.00,2000.000000
2,2,Pro,50,0.85,0.99,50.505051
3,3,Basic,15,0.90,0.08,150.000000
4,4,Basic,15,0.90,1.00,15.000000
...,...,...,...,...,...,...
495,495,Basic,15,0.90,0.00,150.000000
496,496,Basic,15,0.90,0.00,150.000000
497,497,Basic,15,0.90,0.74,20.270270
498,498,Enterprise,200,0.75,0.07,2000.000000


### Quadrant Segmentation

In [48]:
median_cltv = merged_df['projected_cltv'].median()
median_churn_risk = merged_df['churn'].median()
print(median_cltv)
print(median_churn_prob)

150.0
0.02


In [56]:
# seg1 -> Save at All Costs: High CLTV, High Risk
# seg2 -> Monitor & Nurture: High CLTV, Low Risk
# seg3 -> Let Go / Automated Re-engagement: Low CLTV, High Risk
# seg4 -> Low Maintenance: Low CLTV, Low Risk
seg1_df = merged_df[(merged_df['projected_cltv']>median_cltv) & (merged_df['churn']>median_churn_risk)]
seg2_df = merged_df[(merged_df['projected_cltv']>median_cltv) & (merged_df['churn']<median_churn_risk)]
seg3_df = merged_df[(merged_df['projected_cltv']<median_cltv) & (merged_df['churn']>median_churn_risk)]
seg4_df = merged_df[(merged_df['projected_cltv']<median_cltv) & (merged_df['churn']<median_churn_risk)]

In [58]:
seg1_df.describe()

,user_id,monthly_fee,gross_margin_pct,churn,projected_cltv
count,60.000000,60.000000,60.000000,60.000000,60.000000
mean,259.100000,127.500000,0.798333,0.319333,669.701833
std,129.958885,75.590893,0.050394,0.369708,605.200878
min,30.000000,50.000000,0.750000,0.030000,192.307692
25%,155.500000,50.000000,0.750000,0.067500,216.806919
50%,262.000000,200.000000,0.750000,0.125000,500.000000
75%,364.500000,200.000000,0.850000,0.562500,531.250000
max,498.000000,200.000000,0.850000,1.000000,2000.000000


In [59]:
seg2_df.describe()

,user_id,monthly_fee,gross_margin_pct,churn,projected_cltv
count,104.000000,104.000000,104.000000,104.000000,104.000000
mean,255.644231,84.615385,0.826923,0.001154,846.153846
std,149.389562,63.504806,0.042337,0.003210,635.048057
min,1.000000,50.000000,0.750000,0.000000,500.000000
25%,139.250000,50.000000,0.850000,0.000000,500.000000
50%,246.500000,50.000000,0.850000,0.000000,500.000000
75%,396.750000,50.000000,0.850000,0.000000,500.000000
max,499.000000,200.000000,0.850000,0.010000,2000.000000


In [60]:
seg3_df.describe()

,user_id,monthly_fee,gross_margin_pct,churn,projected_cltv
count,135.000000,135.000000,135.000000,135.000000,135.000000
mean,257.103704,25.370370,0.885185,0.697778,47.909301
std,146.905151,16.041337,0.022916,0.325894,33.000335
min,0.000000,15.000000,0.850000,0.110000,15.000000
25%,128.000000,15.000000,0.850000,0.340000,16.129032
50%,262.000000,15.000000,0.900000,0.830000,50.000000
75%,375.500000,50.000000,0.900000,0.990000,65.217391
max,497.000000,50.000000,0.900000,1.000000,136.363636


In [64]:
seg4_df.describe()

,user_id,monthly_fee,gross_margin_pct,churn,projected_cltv
count,0.0,0.0,0.0,0.0,0.0
mean,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN


In [65]:
seg4_df

,user_id,subscription_tier,monthly_fee,gross_margin_pct,churn,projected_cltv


### Budget Allocation

In [67]:
total_seg1_cltv = seg1_df['projected_cltv'].sum()
total_seg1_cltv

np.float64(40182.1099904645)

In [70]:
retention_budget = 0.1*total_seg1_cltv
print(f"The Final Retention budget is ${retention_budget}")

The Final Retention budget is $4018.2109990464505
